# ACA 2 · Detección de neumonía en radiografías de tórax con redes neuronales convolucionales

**Corporación Unificada Nacional de Educación Superior (CUN)** · Escuela de Ingeniería · Especialización en Inteligencia Artificial
**Asignatura:** Aprendizaje Automático · **Docente:** Hamilton Rivera Flor

---

### Pregunta orientadora
¿Cómo aprovechar una red neuronal convolucional preentrenada para distinguir radiografías de tórax **normales** de radiografías con **neumonía**, maximizando la detección de casos positivos y ofreciendo una explicación visual de cada predicción a través de una aplicación en Streamlit?

### Flujo metodológico

| Paso | Contenido en este notebook |
|---|---|
| 1. Exploración y gestión SQL | Inventario de imágenes en **DuckDB**, EDA con consultas SQL, detección de duplicados por hash |
| 2. Preparación y pipeline | Partición **por paciente** (sin fuga de datos), caché en memoria, aumento de datos |
| 3. Modelado neuronal | CNN propia (línea base) vs. **EfficientNet-B0** preentrenada con fine-tuning en dos fases |
| 4. Evaluación | Curvas de pérdida y exactitud, F1, recall, ROC-AUC, matriz de confusión, ajuste de umbral, mapas de activación (CAM) |
| 5. Despliegue | Exportación a **ONNX** + metadatos JSON para la app Streamlit |

> **Ejecución recomendada:** Google Colab con GPU (`Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`). Tiempo total aproximado: 15–25 min.

## 0. Configuración del entorno

In [ ]:
# Dependencias que no vienen por defecto en Colab
!pip install -q duckdb onnx onnxruntime onnxscript kagglehub

In [ ]:
import os, io, re, json, copy, time, hashlib, random, zipfile, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── Parámetros globales (todo lo ajustable está aquí) ──────────────────────
SEMILLA        = 42
TAM_CACHE      = 256          # lado de la imagen guardada en memoria
TAM_ENTRADA    = 224          # lado que recibe la red
BATCH          = 32
EPOCAS_BASE    = 12           # CNN línea base
EPOCAS_FASE1   = 3            # EfficientNet: solo la cabeza
EPOCAS_FASE2   = 12           # EfficientNet: fine-tuning completo
PACIENCIA      = 4            # early stopping
LR_BASE        = 1e-3
LR_FASE1       = 1e-3
LR_FASE2       = 1e-4
CLASES         = ["NORMAL", "NEUMONIA"]   # índice 1 = clase positiva
MEDIA_IMAGENET = [0.485, 0.456, 0.406]
DESV_IMAGENET  = [0.229, 0.224, 0.225]

DIR_TRABAJO    = Path("aca2_trabajo")
DIR_ARTEFACTOS = DIR_TRABAJO / "artefactos"      # lo que consume Streamlit
DIR_FIGURAS    = DIR_ARTEFACTOS / "figuras"      # imágenes para el póster
DIR_EJEMPLOS   = DIR_ARTEFACTOS / "ejemplos"     # radiografías de prueba para la demo
RUTA_DB        = DIR_TRABAJO / "chest_xray.duckdb"
for d in (DIR_ARTEFACTOS, DIR_FIGURAS, DIR_EJEMPLOS):
    d.mkdir(parents=True, exist_ok=True)

def fijar_semillas(semilla: int = SEMILLA) -> None:
    random.seed(semilla); np.random.seed(semilla)
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    torch.backends.cudnn.benchmark = True   # rendimiento > determinismo bit a bit

fijar_semillas()
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = DISPOSITIVO.type == "cuda"
print(f"PyTorch {torch.__version__} · dispositivo: {DISPOSITIVO}"
      + (f" ({torch.cuda.get_device_name(0)})" if USAR_AMP else " — ⚠ sin GPU, el entrenamiento será lento"))

## 1. Exploración y gestión de datos con SQL

### 1.1 Descarga del dataset
**Chest X-Ray Images (Pneumonia)** — Kermany et al. (2018), publicado en Kaggle por Mooney (2018). Radiografías anteroposteriores de pacientes pediátricos de 1 a 5 años del Guangzhou Women and Children's Medical Center.

`kagglehub` descarga datasets públicos sin credenciales. Si falla, suba su `kaggle.json` (Kaggle → Settings → API → *Create New Token*) y vuelva a ejecutar. Si ya tiene el dataset descomprimido, defina la variable de entorno `ACA2_DATA_DIR` con su ruta.

In [ ]:
if os.environ.get("ACA2_DATA_DIR"):
    DIR_DATOS = Path(os.environ["ACA2_DATA_DIR"])
else:
    import kagglehub
    try:
        DIR_DATOS = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
    except Exception as e:
        print("Descarga anónima falló:", e)
        try:
            from google.colab import files
            print("Suba su archivo kaggle.json")
            files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        except ImportError:
            raise RuntimeError("Configure ~/.kaggle/kaggle.json o ACA2_DATA_DIR") from e
        DIR_DATOS = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
print("Dataset en:", DIR_DATOS)

### 1.2 Inventario y caché en una sola pasada
Cada archivo se lee **una vez** para: calcular su hash MD5 (detección de duplicados), registrar dimensiones originales y guardar una versión en escala de grises de 256×256 en memoria (~380 MB).

Dos optimizaciones importantes:
- `Image.draft()` decodifica el JPEG directamente a menor resolución (escalado DCT), varias veces más rápido que decodificar a tamaño completo y luego reducir.
- Lectura paralela con hilos (la E/S y la decodificación liberan el GIL).

**Metadatos extraídos del nombre de archivo:**
- `NORMAL2-IM-1427-0001.jpeg` → paciente `IM-1427`
- `person1_bacteria_1.jpeg` → paciente `person1`, subtipo bacteriana

In [ ]:
PATRON_PERSONA = re.compile(r"(person\d+)_(bacteria|virus)", re.IGNORECASE)
PATRON_NORMAL  = re.compile(r"(IM-\d+)", re.IGNORECASE)

def metadatos_desde_ruta(ruta: Path) -> dict:
    nombre = ruta.name
    carpeta_clase = ruta.parent.name.upper()
    if carpeta_clase == "PNEUMONIA":
        m = PATRON_PERSONA.search(nombre)
        paciente = f"P_{m.group(1).lower()}" if m else f"P_{nombre}"
        subtipo = m.group(2).lower() if m else "desconocido"
        clase = "NEUMONIA"
    else:
        m = PATRON_NORMAL.search(nombre)
        paciente = f"N_{m.group(1).upper()}" if m else f"N_{nombre}"
        subtipo, clase = "normal", "NORMAL"
    return {"ruta": str(ruta), "archivo": nombre, "clase": clase, "subtipo": subtipo,
            "paciente_id": paciente, "split_original": ruta.parent.parent.name.lower()}

def cargar_imagen_gris(origen, tam: int = TAM_CACHE) -> Image.Image:
    # Misma función que usará la app Streamlit: garantiza preprocesamiento idéntico.
    img = Image.open(origen)
    img.draft("L", (tam, tam))
    return img.convert("L").resize((tam, tam), Image.BILINEAR)

def procesar(ruta: Path):
    datos = ruta.read_bytes()
    with Image.open(io.BytesIO(datos)) as im:
        ancho, alto = im.size
    arr = np.asarray(cargar_imagen_gris(io.BytesIO(datos)), dtype=np.uint8)
    return hashlib.md5(datos).hexdigest(), ancho, alto, len(datos), arr

rutas = sorted(p for p in DIR_DATOS.rglob("*")
               if p.suffix.lower() in {".jpeg", ".jpg", ".png"}
               and "__MACOSX" not in p.parts and not p.name.startswith("._")
               and p.parent.name.upper() in {"NORMAL", "PNEUMONIA"})
print(f"Archivos encontrados: {len(rutas):,}")

t0 = time.time()
with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 2) * 2)) as pool:
    resultados = list(pool.map(procesar, rutas))
print(f"Procesados en {time.time() - t0:.1f} s")

inventario = pd.DataFrame([metadatos_desde_ruta(r) for r in rutas])
inventario["hash_md5"] = [r[0] for r in resultados]
inventario["ancho"]    = [r[1] for r in resultados]
inventario["alto"]     = [r[2] for r in resultados]
inventario["bytes"]    = [r[3] for r in resultados]
inventario["idx_cache"] = np.arange(len(inventario))
CACHE = np.stack([r[4] for r in resultados])       # (N, 256, 256) uint8
del resultados
print(f"Caché: {CACHE.shape} · {CACHE.nbytes / 1e6:.0f} MB")
inventario.head()

### 1.3 Base de datos DuckDB
DuckDB es una base analítica embebida (un solo archivo, sin servidor), columnar y muy rápida para agregaciones. El inventario se **inserta** como tabla y todo el EDA se resuelve con **consultas SQL**.

In [ ]:
if RUTA_DB.exists():
    RUTA_DB.unlink()
con = duckdb.connect(str(RUTA_DB))

con.execute('''
    CREATE TABLE imagenes (
        idx_cache       INTEGER PRIMARY KEY,
        ruta            VARCHAR NOT NULL,
        archivo         VARCHAR NOT NULL,
        clase           VARCHAR NOT NULL CHECK (clase IN ('NORMAL', 'NEUMONIA')),
        subtipo         VARCHAR NOT NULL,
        paciente_id     VARCHAR NOT NULL,
        split_original  VARCHAR,
        hash_md5        VARCHAR NOT NULL,
        ancho           INTEGER,
        alto            INTEGER,
        bytes           BIGINT
    )
''')
con.register("df_inventario", inventario)
con.execute('''
    INSERT INTO imagenes
    SELECT idx_cache, ruta, archivo, clase, subtipo, paciente_id, split_original,
           hash_md5, ancho, alto, bytes
    FROM df_inventario
''')
con.unregister("df_inventario")
print(con.sql("SELECT COUNT(*) AS registros FROM imagenes").fetchone()[0], "registros insertados")

In [ ]:
# Duplicados exactos (el dataset de Kaggle trae la carpeta chest_xray repetida)
dup = con.sql('''
    SELECT COUNT(*) AS archivos_totales,
           COUNT(DISTINCT hash_md5) AS imagenes_unicas,
           COUNT(*) - COUNT(DISTINCT hash_md5) AS duplicados
    FROM imagenes
''').df()
display(dup)

# Tabla deduplicada: se conserva la primera aparición de cada hash
con.execute('''
    CREATE TABLE imagenes_unicas AS
    SELECT * FROM imagenes
    QUALIFY ROW_NUMBER() OVER (PARTITION BY hash_md5 ORDER BY idx_cache) = 1
''')

# ¿Hay hashes idénticos con etiquetas distintas? (sería ruido de etiquetado)
conflictos = con.sql('''
    SELECT hash_md5, COUNT(DISTINCT clase) AS clases
    FROM imagenes GROUP BY hash_md5 HAVING COUNT(DISTINCT clase) > 1
''').df()
print("Hashes con etiquetas en conflicto:", len(conflictos))

In [ ]:
distribucion = con.sql('''
    SELECT clase, subtipo,
           COUNT(*)                                        AS imagenes,
           COUNT(DISTINCT paciente_id)                     AS pacientes,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM imagenes_unicas
    GROUP BY clase, subtipo ORDER BY clase, subtipo
''').df()
display(distribucion)

por_paciente = con.sql('''
    SELECT clase,
           ROUND(AVG(n), 2) AS media_img_por_paciente,
           MAX(n)           AS max_img_por_paciente,
           SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS pacientes_con_varias_img
    FROM (SELECT clase, paciente_id, COUNT(*) AS n
          FROM imagenes_unicas GROUP BY clase, paciente_id)
    GROUP BY clase
''').df()
display(por_paciente)

dimensiones = con.sql('''
    SELECT clase,
           ROUND(AVG(ancho)) AS ancho_medio, ROUND(AVG(alto)) AS alto_medio,
           MIN(ancho) AS ancho_min, MAX(ancho) AS ancho_max,
           ROUND(AVG(ancho * 1.0 / alto), 2) AS relacion_aspecto_media,
           ROUND(AVG(bytes) / 1024) AS kb_medio
    FROM imagenes_unicas GROUP BY clase
''').df()
display(dimensiones)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))

clases_df = con.sql("SELECT clase, COUNT(*) n FROM imagenes_unicas GROUP BY clase ORDER BY clase").df()
ejes[0].bar(clases_df["clase"], clases_df["n"], color=["#2a9d8f", "#e76f51"])
for i, v in enumerate(clases_df["n"]):
    ejes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
ejes[0].set_title("Distribución de clases (desbalance)")

sub = con.sql("SELECT subtipo, COUNT(*) n FROM imagenes_unicas GROUP BY subtipo ORDER BY n DESC").df()
ejes[1].pie(sub["n"], labels=sub["subtipo"], autopct="%1.1f%%", colors=["#e76f51", "#2a9d8f", "#f4a261", "#999"])
ejes[1].set_title("Subtipos")

dims = con.sql("SELECT clase, ancho, alto FROM imagenes_unicas").df()
for c, col in zip(CLASES, ["#2a9d8f", "#e76f51"]):
    d = dims[dims.clase == c]
    ejes[2].scatter(d.ancho, d.alto, s=4, alpha=0.4, label=c, color=col)
ejes[2].set_xlabel("ancho (px)"); ejes[2].set_ylabel("alto (px)"); ejes[2].legend()
ejes[2].set_title("Dimensiones originales")
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "eda_distribucion.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Muestra visual por subtipo
fig, ejes = plt.subplots(3, 5, figsize=(15, 9))
for fila, subtipo in enumerate(["normal", "bacteria", "virus"]):
    idx = con.sql(f'''SELECT idx_cache FROM imagenes_unicas WHERE subtipo = '{subtipo}'
                      USING SAMPLE 5 ROWS (reservoir, {SEMILLA})''').df()["idx_cache"]
    for col, i in enumerate(idx):
        ejes[fila, col].imshow(CACHE[i], cmap="gray"); ejes[fila, col].axis("off")
    ejes[fila, 0].set_title(subtipo.upper(), loc="left", fontweight="bold")
plt.suptitle("Radiografías de ejemplo (256×256, escala de grises)")
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "eda_muestras.png", dpi=120, bbox_inches="tight"); plt.show()

# Intensidad media por clase: ¿hay diferencias globales de brillo que el modelo podría usar como atajo?
unicas = con.sql("SELECT idx_cache, clase FROM imagenes_unicas").df()
unicas["brillo_medio"] = CACHE[unicas.idx_cache.values].mean(axis=(1, 2))
display(unicas.groupby("clase")["brillo_medio"].describe().round(1))

**Hallazgos del EDA:**
1. El dataset de Kaggle trae carpetas duplicadas; la deduplicación por hash evita que una misma imagen quede en entrenamiento y en prueba.
2. Desbalance cercano a **3:1** a favor de neumonía → se compensará con pesos de clase en la función de pérdida y se priorizarán F1 y recall sobre accuracy.
3. Hay pacientes con **varias radiografías**. Si se parte por imagen, el modelo puede "memorizar" al paciente y las métricas de prueba quedan infladas (*data leakage*). Por eso la partición se hace **por paciente**.
4. La partición `val` original tiene solo 16 imágenes, insuficiente para seleccionar modelos; se rehace la partición completa.
5. Las dimensiones y relaciones de aspecto varían mucho → se normaliza a un tamaño fijo.

## 2. Preparación y pipeline

### 2.1 Partición estratificada por paciente (70 / 15 / 15)
`StratifiedGroupKFold` garantiza dos cosas a la vez: **ningún paciente aparece en dos particiones** (grupos) y la proporción de clases se mantiene similar en cada una (estratificación).

In [ ]:
df = con.sql("SELECT * FROM imagenes_unicas ORDER BY idx_cache").df()

# Prueba ≈ 1/7 (14,3 %)
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEMILLA)
idx_resto, idx_prueba = next(sgkf.split(df, df["clase"], groups=df["paciente_id"]))
resto = df.iloc[idx_resto]
# Validación ≈ 1/6 del resto (14,3 % del total)
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEMILLA)
i_ent, i_val = next(sgkf2.split(resto, resto["clase"], groups=resto["paciente_id"]))

df["split"] = "entrenamiento"
df.loc[resto.index[i_val], "split"] = "validacion"
df.loc[df.index[idx_prueba], "split"] = "prueba"
df["etiqueta"] = (df["clase"] == "NEUMONIA").astype(int)

# Persistir la partición en la base de datos
con.register("df_split", df[["idx_cache", "split"]])
con.execute("CREATE OR REPLACE TABLE particiones AS SELECT * FROM df_split")
con.unregister("df_split")
con.execute('''
    CREATE OR REPLACE VIEW v_dataset AS
    SELECT u.*, p.split FROM imagenes_unicas u JOIN particiones p USING (idx_cache)
''')

resumen_split = con.sql('''
    SELECT split,
           COUNT(*) AS imagenes,
           COUNT(DISTINCT paciente_id) AS pacientes,
           SUM(CASE WHEN clase = 'NEUMONIA' THEN 1 ELSE 0 END) AS neumonia,
           SUM(CASE WHEN clase = 'NORMAL' THEN 1 ELSE 0 END)   AS normal,
           ROUND(100.0 * SUM(CASE WHEN clase = 'NEUMONIA' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_neumonia
    FROM v_dataset GROUP BY split ORDER BY imagenes DESC
''').df()
display(resumen_split)

# Verificación de no fuga: ningún paciente en más de una partición
fuga = con.sql('''
    SELECT COUNT(*) FROM (
        SELECT paciente_id FROM v_dataset GROUP BY paciente_id HAVING COUNT(DISTINCT split) > 1)
''').fetchone()[0]
assert fuga == 0, f"{fuga} pacientes aparecen en varias particiones"
print("✔ Sin fuga de pacientes entre particiones")

### 2.2 Transformaciones y aumento de datos
| Transformación | Justificación clínica / técnica |
|---|---|
| `RandomResizedCrop` (escala 0,80–1,0) | Variaciones de encuadre y distancia del equipo |
| `RandomRotation(±10°)` | Posicionamiento imperfecto del paciente pediátrico |
| `ColorJitter` (brillo y contraste ±20 %) | Diferencias de exposición entre equipos de rayos X |
| `RandomAffine` (traslación ±5 %) | Centrado variable del tórax |
| **Sin volteo horizontal** | El corazón está a la izquierda (*situs solitus*); voltear crearía anatomía irreal (dextrocardia) |
| `Grayscale(3)` + normalización ImageNet | La red preentrenada espera 3 canales con esa estadística |

Validación y prueba solo se redimensionan: el aumento se aplica únicamente al entrenamiento.

In [ ]:
tf_entrenamiento = transforms.Compose([
    transforms.RandomResizedCrop(TAM_ENTRADA, scale=(0.80, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_IMAGENET, DESV_IMAGENET),
])
tf_evaluacion = transforms.Compose([
    transforms.Resize((TAM_ENTRADA, TAM_ENTRADA)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_IMAGENET, DESV_IMAGENET),
])

class RadiografiasEnMemoria(Dataset):
    # Lee desde la caché uint8: sin acceso a disco durante el entrenamiento.
    def __init__(self, marco: pd.DataFrame, transformacion):
        self.indices = marco["idx_cache"].to_numpy()
        self.etiquetas = marco["etiqueta"].to_numpy()
        self.tf = transformacion
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img = Image.fromarray(CACHE[self.indices[i]], mode="L")
        return self.tf(img), int(self.etiquetas[i])

particion = {s: df[df.split == s].reset_index(drop=True) for s in ["entrenamiento", "validacion", "prueba"]}
NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() > 1 else 0
cargadores = {
    s: DataLoader(RadiografiasEnMemoria(m, tf_entrenamiento if s == "entrenamiento" else tf_evaluacion),
                  batch_size=BATCH, shuffle=(s == "entrenamiento"),
                  num_workers=NUM_WORKERS, pin_memory=USAR_AMP, drop_last=False,
                  persistent_workers=NUM_WORKERS > 0)
    for s, m in particion.items()
}

# Pesos de clase inversos a la frecuencia (solo con datos de entrenamiento)
conteo = np.bincount(particion["entrenamiento"]["etiqueta"], minlength=2)
PESOS_CLASE = torch.tensor(conteo.sum() / (2 * conteo), dtype=torch.float32)
print("Imágenes por clase en entrenamiento:", dict(zip(CLASES, conteo)))
print("Pesos de clase:", dict(zip(CLASES, PESOS_CLASE.numpy().round(3))))

In [ ]:
# Visualizar el efecto del aumento de datos sobre una misma radiografía
desnormalizar = lambda t: (t * torch.tensor(DESV_IMAGENET)[:, None, None]
                           + torch.tensor(MEDIA_IMAGENET)[:, None, None]).clamp(0, 1)
muestra = Image.fromarray(CACHE[particion["entrenamiento"].idx_cache.iloc[0]], mode="L")
fig, ejes = plt.subplots(1, 6, figsize=(18, 3.4))
ejes[0].imshow(tf_evaluacion(muestra)[0] * DESV_IMAGENET[0] + MEDIA_IMAGENET[0], cmap="gray")
ejes[0].set_title("Original"); ejes[0].axis("off")
for k in range(1, 6):
    ejes[k].imshow(desnormalizar(tf_entrenamiento(muestra))[0], cmap="gray")
    ejes[k].set_title(f"Aumento {k}"); ejes[k].axis("off")
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "aumento_datos.png", dpi=120, bbox_inches="tight"); plt.show()

## 3. Modelado con redes neuronales

Se comparan dos arquitecturas para justificar técnicamente la elección final:

**A. CNN propia (línea base)** — 4 bloques `Conv → BatchNorm → ReLU → MaxPool` con 32, 64, 128 y 256 filtros, *Global Average Pooling* y capa densa. Entrenada desde cero.

**B. EfficientNet-B0 preentrenada en ImageNet (transfer learning)** — Tan & Le (2019). Escala de forma balanceada profundidad, anchura y resolución; usa bloques MBConv (convoluciones separables en profundidad + *squeeze-and-excitation*). Con ~4 M de parámetros es liviana para desplegar. Entrenamiento en dos fases:
1. **Fase 1:** extractor congelado, solo se entrena la cabeza clasificadora (evita destruir los filtros preentrenados con gradientes grandes de una capa aleatoria).
2. **Fase 2:** se descongela toda la red y se ajusta con tasa de aprendizaje 10 veces menor y decaimiento coseno.

Ambas usan entropía cruzada **ponderada por clase**, optimizador AdamW, precisión mixta (AMP) en GPU y *early stopping* sobre el F1 macro de validación.

In [ ]:
class CNNBase(nn.Module):
    def __init__(self, n_clases: int = 2):
        super().__init__()
        def bloque(entrada, salida):
            return nn.Sequential(
                nn.Conv2d(entrada, salida, 3, padding=1, bias=False), nn.BatchNorm2d(salida), nn.ReLU(inplace=True),
                nn.Conv2d(salida, salida, 3, padding=1, bias=False), nn.BatchNorm2d(salida), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(bloque(3, 32), bloque(32, 64), bloque(64, 128), bloque(128, 256))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(256, n_clases))
    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)).flatten(1))

def crear_efficientnet(n_clases: int = 2, preentrenado: bool = True) -> nn.Module:
    pesos = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if preentrenado else None
    m = models.efficientnet_b0(weights=pesos)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, n_clases))
    return m

contar = lambda m: sum(p.numel() for p in m.parameters())
print(f"CNN base:        {contar(CNNBase()):>10,} parámetros")
print(f"EfficientNet-B0: {contar(crear_efficientnet(preentrenado=False)):>10,} parámetros")

In [ ]:
@torch.no_grad()
def predecir(modelo: nn.Module, cargador: DataLoader):
    # Devuelve (etiquetas reales, probabilidad de NEUMONIA, pérdida media).
    modelo.eval()
    criterio = nn.CrossEntropyLoss(weight=PESOS_CLASE.to(DISPOSITIVO))
    reales, probs, perdida_total = [], [], 0.0
    for x, y in cargador:
        x, y = x.to(DISPOSITIVO, non_blocking=True), y.to(DISPOSITIVO, non_blocking=True)
        with torch.autocast(device_type=DISPOSITIVO.type, enabled=USAR_AMP):
            logits = modelo(x)
        perdida_total += criterio(logits.float(), y).item() * len(y)
        probs.append(torch.softmax(logits.float(), 1)[:, 1].cpu()); reales.append(y.cpu())
    return torch.cat(reales).numpy(), torch.cat(probs).numpy(), perdida_total / len(cargador.dataset)

def entrenar(modelo, epocas, lr, nombre, historia=None, parametros=None, congelar_bn_features=False):
    # Bucle genérico con AMP, scheduler coseno y early stopping por F1 macro de validación.
    historia = historia if historia is not None else []
    parametros = parametros if parametros is not None else [p for p in modelo.parameters() if p.requires_grad]
    optimizador = torch.optim.AdamW(parametros, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=epocas)
    criterio = nn.CrossEntropyLoss(weight=PESOS_CLASE.to(DISPOSITIVO))
    escalador = torch.amp.GradScaler(enabled=USAR_AMP)
    mejor_f1 = max((h["val_f1"] for h in historia), default=-1.0)
    mejor_estado, sin_mejora = copy.deepcopy(modelo.state_dict()), 0

    for epoca in range(1, epocas + 1):
        t0 = time.time()
        modelo.train()
        if congelar_bn_features:          # BatchNorm del extractor congelado no debe actualizar estadísticas
            modelo.features.eval()
        perdida, aciertos, n = 0.0, 0, 0
        for x, y in cargadores["entrenamiento"]:
            x, y = x.to(DISPOSITIVO, non_blocking=True), y.to(DISPOSITIVO, non_blocking=True)
            optimizador.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DISPOSITIVO.type, enabled=USAR_AMP):
                logits = modelo(x)
                loss = criterio(logits.float(), y)
            escalador.scale(loss).backward()
            escalador.step(optimizador); escalador.update()
            perdida += loss.item() * len(y); aciertos += (logits.argmax(1) == y).sum().item(); n += len(y)
        scheduler.step()

        y_val, p_val, loss_val = predecir(modelo, cargadores["validacion"])
        pred_val = (p_val >= 0.5).astype(int)
        registro = {"modelo": nombre, "epoca": len(historia) + 1,
                    "train_loss": perdida / n, "train_acc": aciertos / n,
                    "val_loss": loss_val, "val_acc": accuracy_score(y_val, pred_val),
                    "val_f1": f1_score(y_val, pred_val, average="macro"),
                    "val_auc": roc_auc_score(y_val, p_val),
                    "lr": optimizador.param_groups[0]["lr"]}
        historia.append(registro)
        marca = ""
        if registro["val_f1"] > mejor_f1:
            mejor_f1, mejor_estado, sin_mejora, marca = registro["val_f1"], copy.deepcopy(modelo.state_dict()), 0, " ★"
        else:
            sin_mejora += 1
        print(f"[{nombre}] época {registro['epoca']:>2} | loss {registro['train_loss']:.4f} / {loss_val:.4f} "
              f"| acc {registro['train_acc']:.3f} / {registro['val_acc']:.3f} | F1 val {registro['val_f1']:.4f} "
              f"| AUC val {registro['val_auc']:.4f} | {time.time() - t0:.0f}s{marca}")
        if sin_mejora >= PACIENCIA:
            print(f"Early stopping: {PACIENCIA} épocas sin mejora"); break

    modelo.load_state_dict(mejor_estado)
    return modelo, historia

### 3.1 Modelo A — CNN desde cero (línea base)

In [ ]:
fijar_semillas()
modelo_base = CNNBase().to(DISPOSITIVO)
modelo_base, hist_base = entrenar(modelo_base, EPOCAS_BASE, LR_BASE, "CNN base")

### 3.2 Modelo B — EfficientNet-B0 con transfer learning

In [ ]:
fijar_semillas()
modelo_eff = crear_efficientnet().to(DISPOSITIVO)

# Fase 1: extractor congelado
for p in modelo_eff.features.parameters():
    p.requires_grad = False
print(f"Fase 1 · parámetros entrenables: {sum(p.numel() for p in modelo_eff.parameters() if p.requires_grad):,}")
modelo_eff, hist_eff = entrenar(modelo_eff, EPOCAS_FASE1, LR_FASE1, "EffNet-B0",
                                congelar_bn_features=True)
EPOCA_FIN_FASE1 = len(hist_eff)

# Fase 2: fine-tuning completo
for p in modelo_eff.parameters():
    p.requires_grad = True
print(f"Fase 2 · parámetros entrenables: {contar(modelo_eff):,}")
modelo_eff, hist_eff = entrenar(modelo_eff, EPOCAS_FASE2, LR_FASE2, "EffNet-B0", historia=hist_eff)

## 4. Evaluación y métricas

### 4.1 Curvas de aprendizaje
La línea punteada marca el paso de la fase 1 a la fase 2 en EfficientNet.

In [ ]:
historia = pd.DataFrame(hist_base + hist_eff)
fig, ejes = plt.subplots(1, 3, figsize=(18, 4.5))
for nombre, color in [("CNN base", "#264653"), ("EffNet-B0", "#e76f51")]:
    h = historia[historia.modelo == nombre]
    ejes[0].plot(h.epoca, h.train_loss, "--", color=color, label=f"{nombre} · entrenamiento")
    ejes[0].plot(h.epoca, h.val_loss, "-o", ms=3, color=color, label=f"{nombre} · validación")
    ejes[1].plot(h.epoca, h.train_acc, "--", color=color)
    ejes[1].plot(h.epoca, h.val_acc, "-o", ms=3, color=color, label=nombre)
    ejes[2].plot(h.epoca, h.val_f1, "-o", ms=3, color=color, label=f"{nombre} · F1")
    ejes[2].plot(h.epoca, h.val_auc, ":", color=color, label=f"{nombre} · AUC")
for ax, t in zip(ejes, ["Pérdida (entropía cruzada ponderada)", "Exactitud", "F1 macro y AUC (validación)"]):
    ax.axvline(EPOCA_FIN_FASE1 + 0.5, color="gray", ls=":", lw=1)
    ax.set_title(t); ax.set_xlabel("época"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "curvas_aprendizaje.png", dpi=150, bbox_inches="tight"); plt.show()

### 4.2 Selección del umbral de decisión (solo con validación)
Un umbral de 0,5 no es necesariamente óptimo con clases desbalanceadas. Se barre el umbral en **validación** y se elige el que maximiza el F1 macro; el conjunto de prueba **no** se usa para esta decisión, de modo que su evaluación siga siendo imparcial.

In [ ]:
def barrer_umbral(y, p):
    umbrales = np.round(np.arange(0.05, 0.96, 0.01), 2)
    filas = [{"umbral": u,
              "f1_macro": f1_score(y, (p >= u).astype(int), average="macro"),
              "recall_neumonia": recall_score(y, (p >= u).astype(int)),
              "especificidad": recall_score(y, (p >= u).astype(int), pos_label=0)} for u in umbrales]
    return pd.DataFrame(filas)

y_val, p_val_eff, _ = predecir(modelo_eff, cargadores["validacion"])
barrido = barrer_umbral(y_val, p_val_eff)
UMBRAL = float(barrido.loc[barrido.f1_macro.idxmax(), "umbral"])

plt.figure(figsize=(8, 4))
for col, c in [("f1_macro", "#264653"), ("recall_neumonia", "#e76f51"), ("especificidad", "#2a9d8f")]:
    plt.plot(barrido.umbral, barrido[col], label=col, color=c)
plt.axvline(UMBRAL, color="k", ls="--", label=f"umbral elegido = {UMBRAL:.2f}")
plt.xlabel("umbral sobre P(neumonía)"); plt.title("Barrido de umbral en validación (EffNet-B0)"); plt.legend()
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "barrido_umbral.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"Umbral operativo: {UMBRAL:.2f}")

### 4.3 Evaluación final en el conjunto de prueba

In [ ]:
def metricas(y, p, umbral):
    pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall_sensibilidad": recall_score(y, pred),
            "especificidad": tn / (tn + fp) if (tn + fp) else 0.0,
            "f1_neumonia": f1_score(y, pred),
            "f1_macro": f1_score(y, pred, average="macro"),
            "roc_auc": roc_auc_score(y, p),
            "vp": int(tp), "fp": int(fp), "vn": int(tn), "fn": int(fn)}

y_test, p_test_base, loss_test_base = predecir(modelo_base, cargadores["prueba"])
_,      p_test_eff,  loss_test_eff  = predecir(modelo_eff,  cargadores["prueba"])
y_val_b, p_val_base, _ = predecir(modelo_base, cargadores["validacion"])
b_base = barrer_umbral(y_val_b, p_val_base)
UMBRAL_BASE = float(b_base.loc[b_base.f1_macro.idxmax(), "umbral"])

resultados = {
    "CNN base (umbral 0,50)":                  metricas(y_test, p_test_base, 0.5),
    f"CNN base (umbral {UMBRAL_BASE:.2f})":    metricas(y_test, p_test_base, UMBRAL_BASE),
    "EffNet-B0 (umbral 0,50)":                 metricas(y_test, p_test_eff, 0.5),
    f"EffNet-B0 (umbral {UMBRAL:.2f})":        metricas(y_test, p_test_eff, UMBRAL),
}
tabla = pd.DataFrame(resultados).T
display(tabla.style.format({c: "{:.4f}" for c in tabla.columns if c not in ("vp", "fp", "vn", "fn")})
        .highlight_max(subset=["f1_macro", "roc_auc", "recall_sensibilidad"], color="#d8f3dc"))
tabla.to_csv(DIR_ARTEFACTOS / "comparacion_modelos.csv")

print("\nReporte de clasificación — EffNet-B0, umbral operativo")
print(classification_report(y_test, (p_test_eff >= UMBRAL).astype(int), target_names=CLASES, digits=4))

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nombre, p, u) in zip(ejes[:2], [("CNN base", p_test_base, UMBRAL_BASE), ("EffNet-B0", p_test_eff, UMBRAL)]):
    cm = confusion_matrix(y_test, (p >= u).astype(int))
    etiquetas = np.array([[f"{v}\n({v / cm[i].sum():.1%})" for v in fila] for i, fila in enumerate(cm)])
    sns.heatmap(cm, annot=etiquetas, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASES, yticklabels=CLASES, annot_kws={"size": 13})
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title(f"Matriz de confusión · {nombre} (umbral {u:.2f})")

for nombre, p, c in [("CNN base", p_test_base, "#264653"), ("EffNet-B0", p_test_eff, "#e76f51")]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ejes[2].plot(fpr, tpr, color=c, lw=2, label=f"{nombre} (AUC = {roc_auc_score(y_test, p):.3f})")
ejes[2].plot([0, 1], [0, 1], "k--", lw=1)
ejes[2].set_xlabel("1 − especificidad"); ejes[2].set_ylabel("Sensibilidad"); ejes[2].set_title("Curva ROC (prueba)")
ejes[2].legend()
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "matriz_confusion_roc.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Desempeño por subtipo: ¿se detecta igual la neumonía viral que la bacteriana?
prueba = particion["prueba"].copy()
prueba["p_neumonia"] = p_test_eff
prueba["pred"] = (p_test_eff >= UMBRAL).astype(int)
prueba["acierto"] = (prueba.pred == prueba.etiqueta).astype(int)
con.register("df_pred", prueba[["idx_cache", "p_neumonia", "pred", "acierto"]])
con.execute("CREATE OR REPLACE TABLE predicciones_prueba AS SELECT * FROM df_pred")
con.unregister("df_pred")
por_subtipo = con.sql('''
    SELECT d.subtipo, COUNT(*) AS imagenes,
           ROUND(AVG(p.acierto), 4) AS tasa_acierto,
           ROUND(AVG(p.p_neumonia), 4) AS prob_media_neumonia
    FROM predicciones_prueba p JOIN v_dataset d USING (idx_cache)
    GROUP BY d.subtipo ORDER BY d.subtipo
''').df()
display(por_subtipo)

### 4.4 Explicabilidad: mapas de activación de clase (CAM)
EfficientNet-B0 termina en *Global Average Pooling* seguido de una única capa lineal. Para esa estructura, el mapa de activación de clase de Zhou et al. (2016) se calcula exactamente como

$$\text{CAM}_c(i,j) = \sum_k w_{c,k}\, A_k(i,j)$$

donde $A_k$ son los 1 280 mapas de la última capa convolucional y $w_{c,k}$ los pesos de la capa lineal. En este caso **Grad-CAM** (Selvaraju et al., 2017) da el mismo mapa salvo una constante, porque el gradiente de la salida respecto a $A_k$ es $w_{c,k}/(H\cdot W)$. La celda siguiente lo verifica numéricamente.

Ventaja práctica: la app solo necesita los mapas $A_k$ (una salida extra del modelo ONNX) y los pesos $w$, sin calcular gradientes, por lo que funciona con `onnxruntime` puro.

In [ ]:
class ModeloConMapas(nn.Module):
    # Envoltorio para despliegue: devuelve logits y mapas de la última capa convolucional.
    def __init__(self, base: nn.Module):
        super().__init__()
        self.features, self.avgpool, self.classifier = base.features, base.avgpool, base.classifier
    def forward(self, x):
        mapas = self.features(x)
        return self.classifier(self.avgpool(mapas).flatten(1)), mapas

PESOS_CAM = modelo_eff.classifier[1].weight.detach().cpu().numpy()     # (2, 1280)

def cam_desde_mapas(mapas: np.ndarray, clase: int = 1, tam: int = TAM_ENTRADA) -> np.ndarray:
    cam = np.tensordot(PESOS_CAM[clase], mapas, axes=(0, 0))             # (7, 7)
    cam = np.maximum(cam, 0)
    cam = np.asarray(Image.fromarray(cam.astype(np.float32)).resize((tam, tam), Image.BILINEAR))
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def gradcam_hooks(modelo, x, clase=1):
    # Grad-CAM clásico con gradientes (solo para verificar la equivalencia).
    activ, grads = {}, {}
    capa = modelo.features[-1]
    h1 = capa.register_forward_hook(lambda m, i, o: activ.setdefault("a", o))
    h2 = capa.register_full_backward_hook(lambda m, gi, go: grads.setdefault("g", go[0]))
    modelo.eval(); modelo.zero_grad()
    modelo(x)[0, clase].backward()
    h1.remove(); h2.remove()
    pesos = grads["g"].mean(dim=(2, 3), keepdim=True)
    return F.relu((pesos * activ["a"]).sum(1))[0].detach().cpu().numpy()

envoltorio = ModeloConMapas(modelo_eff).eval()
x_ej = tf_evaluacion(Image.fromarray(CACHE[prueba.idx_cache.iloc[0]], mode="L")).unsqueeze(0).to(DISPOSITIVO)
with torch.no_grad():
    _, mapas_ej = envoltorio(x_ej)
cam_a = np.maximum(np.tensordot(PESOS_CAM[1], mapas_ej[0].cpu().numpy(), axes=(0, 0)), 0)
cam_b = gradcam_hooks(modelo_eff, x_ej.clone().requires_grad_(True))
corr = np.corrcoef(cam_a.ravel(), cam_b.ravel())[0, 1] if cam_a.std() > 0 and cam_b.std() > 0 else 1.0
print(f"Correlación CAM vs Grad-CAM: {corr:.6f}  (≈ 1 confirma la equivalencia)")

In [ ]:
def ejemplos_por_caso(n=3):
    casos = {"Verdadero positivo": (prueba.etiqueta == 1) & (prueba.pred == 1),
             "Verdadero negativo": (prueba.etiqueta == 0) & (prueba.pred == 0),
             "Falso positivo":     (prueba.etiqueta == 0) & (prueba.pred == 1),
             "Falso negativo":     (prueba.etiqueta == 1) & (prueba.pred == 0)}
    return {k: prueba[m].sample(min(n, m.sum()), random_state=SEMILLA) for k, m in casos.items()}

casos = ejemplos_por_caso()
fig, ejes = plt.subplots(4, 3, figsize=(11, 15))
for fila, (titulo, sub) in enumerate(casos.items()):
    for col in range(3):
        ax = ejes[fila, col]; ax.axis("off")
        if col >= len(sub):
            continue
        r = sub.iloc[col]
        img = Image.fromarray(CACHE[r.idx_cache], mode="L")
        with torch.no_grad():
            _, m = envoltorio(tf_evaluacion(img).unsqueeze(0).to(DISPOSITIVO))
        ax.imshow(img.resize((TAM_ENTRADA, TAM_ENTRADA)), cmap="gray")
        ax.imshow(cam_desde_mapas(m[0].float().cpu().numpy()), cmap="jet", alpha=0.35)
        ax.set_title(f"{titulo}\nP(neumonía) = {r.p_neumonia:.2f} · {r.subtipo}", fontsize=9)
plt.suptitle("Mapas de activación (zonas que impulsan la predicción de neumonía)", y=1.0)
plt.tight_layout(); plt.savefig(DIR_FIGURAS / "mapas_activacion.png", dpi=130, bbox_inches="tight"); plt.show()

**Lectura clínica de los mapas:** en los verdaderos positivos la activación debería concentrarse en los campos pulmonares (opacidades, consolidaciones). Si se observan activaciones en bordes, marcas de texto o electrodos, el modelo podría estar usando atajos (*shortcut learning*); esa revisión cualitativa complementa las métricas y se discute en la sustentación.

## 5. Exportación para Streamlit

Se exporta el modelo a **ONNX** (formato abierto e independiente del framework). La app usará `onnxruntime` (~15 MB) en lugar de PyTorch (~800 MB), lo que hace viable el despliegue en Streamlit Community Cloud.

**Artefactos generados:**

| Archivo | Uso en la app |
|---|---|
| `neumonia_effnetb0.onnx` | Modelo con dos salidas: `logits` y `mapas` |
| `pesos_cam.npy` | Pesos $w$ para calcular el CAM |
| `metadatos.json` | Clases, umbral, preprocesamiento, métricas, curvas, matriz de confusión, ROC y estadísticas del dataset |
| `comparacion_modelos.csv` | Tabla de métricas de ambos modelos |
| `figuras/*.png` | Gráficas para el póster |
| `ejemplos/*.jpeg` | Radiografías de prueba para la demostración en vivo |

In [ ]:
import onnx, onnxruntime as ort

RUTA_ONNX = DIR_ARTEFACTOS / "neumonia_effnetb0.onnx"
envoltorio_cpu = ModeloConMapas(copy.deepcopy(modelo_eff).float().cpu()).eval()
x_dummy = torch.randn(1, 3, TAM_ENTRADA, TAM_ENTRADA)

def exportar_onnx():
    # TorchScript exporter con eje de batch dinámico; si la versión de PyTorch ya no lo
    # soporta, se usa el exportador dynamo (requiere onnxscript).
    try:
        torch.onnx.export(envoltorio_cpu, x_dummy, str(RUTA_ONNX), dynamo=False,
                          input_names=["entrada"], output_names=["logits", "mapas"],
                          dynamic_axes={"entrada": {0: "batch"}, "logits": {0: "batch"}, "mapas": {0: "batch"}},
                          opset_version=17, do_constant_folding=True)
        return "torchscript"
    except Exception as e:
        print("Exportador TorchScript no disponible, usando dynamo:", type(e).__name__)
        prog = torch.onnx.export(envoltorio_cpu, (x_dummy,), dynamo=True,
                                 input_names=["entrada"], output_names=["logits", "mapas"])
        prog.save(str(RUTA_ONNX))
        return "dynamo"

metodo = exportar_onnx()
onnx.checker.check_model(onnx.load(str(RUTA_ONNX)))

# Validar que ONNX Runtime reproduce a PyTorch
sesion = ort.InferenceSession(str(RUTA_ONNX), providers=["CPUExecutionProvider"])
lote = torch.stack([tf_evaluacion(Image.fromarray(CACHE[i], mode="L")) for i in prueba.idx_cache.iloc[:1]])
with torch.no_grad():
    logits_pt, mapas_pt = envoltorio_cpu(lote)
logits_ort, mapas_ort = sesion.run(None, {"entrada": lote.numpy()})
dif = max(np.abs(logits_ort - logits_pt.numpy()).max(), np.abs(mapas_ort - mapas_pt.numpy()).max())
print(f"Exportado con {metodo} · {RUTA_ONNX.stat().st_size / 1e6:.1f} MB · diferencia máx. PyTorch vs ONNX: {dif:.2e}")
assert dif < 1e-3, "La salida ONNX difiere de PyTorch"

np.save(DIR_ARTEFACTOS / "pesos_cam.npy", PESOS_CAM.astype(np.float32))

In [ ]:
def a_python(o):
    if isinstance(o, dict):  return {k: a_python(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [a_python(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return a_python(o.tolist())
    return o

fpr_e, tpr_e, _ = roc_curve(y_test, p_test_eff)
fpr_b, tpr_b, _ = roc_curve(y_test, p_test_base)
submuestrear = lambda a, n=150: a[np.linspace(0, len(a) - 1, min(n, len(a))).astype(int)]

metadatos = {
    "proyecto": "ACA 2 · Detección de neumonía en radiografías de tórax",
    "fecha_entrenamiento": time.strftime("%Y-%m-%d %H:%M"),
    "entorno": {"pytorch": torch.__version__, "dispositivo": str(DISPOSITIVO), "exportador_onnx": metodo},
    "modelo": {"arquitectura": "EfficientNet-B0 (ImageNet) + cabeza Dropout(0.3) → Linear(1280, 2)",
               "archivo": RUTA_ONNX.name, "parametros": contar(modelo_eff),
               "entrada": "entrada", "salidas": ["logits", "mapas"],
               "forma_mapas": list(mapas_ort.shape[1:])},
    "clases": CLASES, "clase_positiva": 1, "umbral": UMBRAL, "umbral_base": UMBRAL_BASE,
    "preprocesamiento": {"pasos": ["escala de grises (L)", f"redimensionar a {TAM_CACHE}×{TAM_CACHE} bilineal",
                                   f"redimensionar a {TAM_ENTRADA}×{TAM_ENTRADA} bilineal",
                                   "replicar a 3 canales", "escalar a [0, 1]", "normalizar ImageNet"],
                         "tam_cache": TAM_CACHE, "tam_entrada": TAM_ENTRADA,
                         "media": MEDIA_IMAGENET, "desviacion": DESV_IMAGENET},
    "entrenamiento": {"semilla": SEMILLA, "batch": BATCH, "optimizador": "AdamW (weight_decay=1e-4)",
                      "scheduler": "CosineAnnealingLR", "perdida": "CrossEntropy ponderada por clase",
                      "pesos_clase": PESOS_CLASE.tolist(), "epoca_fin_fase1": EPOCA_FIN_FASE1,
                      "lr": {"base": LR_BASE, "fase1": LR_FASE1, "fase2": LR_FASE2}, "paciencia": PACIENCIA,
                      "aumento_datos": ["RandomResizedCrop(0.80–1.0)", "RandomRotation(±10°)",
                                        "RandomAffine(traslación ±5 %)", "ColorJitter(brillo/contraste ±20 %)",
                                        "sin volteo horizontal"]},
    "dataset": {"nombre": "Chest X-Ray Images (Pneumonia)",
                "fuente": "https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia",
                "archivos_totales": int(dup.archivos_totales[0]), "imagenes_unicas": int(dup.imagenes_unicas[0]),
                "duplicados": int(dup.duplicados[0]),
                "distribucion": distribucion.to_dict("records"), "particiones": resumen_split.to_dict("records")},
    "metricas_prueba": {k: v for k, v in resultados.items()},
    "metricas_principal": metricas(y_test, p_test_eff, UMBRAL),
    "metricas_base": metricas(y_test, p_test_base, UMBRAL_BASE),
    "parametros_base": contar(modelo_base),
    "metricas_por_subtipo": por_subtipo.to_dict("records"),
    "historia": historia.to_dict("records"),
    "roc": {"EffNet-B0": {"fpr": submuestrear(fpr_e), "tpr": submuestrear(tpr_e)},
            "CNN base":  {"fpr": submuestrear(fpr_b), "tpr": submuestrear(tpr_b)}},
    "barrido_umbral": barrido.to_dict("records"),
    "matriz_confusion": confusion_matrix(y_test, (p_test_eff >= UMBRAL).astype(int)).tolist(),
}
with open(DIR_ARTEFACTOS / "metadatos.json", "w", encoding="utf-8") as f:
    json.dump(a_python(metadatos), f, ensure_ascii=False, indent=2)

# Radiografías de prueba (nunca vistas en entrenamiento) para la demo en vivo
for subtipo in ["normal", "bacteria", "virus"]:
    sub = prueba[prueba.subtipo == subtipo].sample(min(3, (prueba.subtipo == subtipo).sum()), random_state=SEMILLA)
    for k, r in enumerate(sub.itertuples(), 1):
        Image.fromarray(CACHE[r.idx_cache], mode="L").save(DIR_EJEMPLOS / f"{subtipo}_{k}.jpeg", quality=92)

con.close()
print("Artefactos:")
for p in sorted(DIR_ARTEFACTOS.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(DIR_ARTEFACTOS)}  ({p.stat().st_size / 1e3:,.0f} KB)")

In [ ]:
# Empaquetar artefactos (y la base DuckDB) para descargar y subir al repositorio de la app
RUTA_ZIP = Path("artefactos_aca2.zip")
with zipfile.ZipFile(RUTA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in DIR_ARTEFACTOS.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(DIR_ARTEFACTOS.parent))
    z.write(RUTA_DB, RUTA_DB.name)
print(f"{RUTA_ZIP} · {RUTA_ZIP.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(str(RUTA_ZIP))
except ImportError:
    pass

## 6. Conclusiones (completar con los valores obtenidos)

1. **Transfer learning vs. entrenamiento desde cero:** EfficientNet-B0 preentrenada supera a la CNN base en F1 macro y AUC con menos épocas, porque reutiliza filtros de bordes y texturas aprendidos en ImageNet; esto es decisivo con apenas unos miles de imágenes.
2. **Rigor en la partición:** la deduplicación por hash y la partición por paciente evitan fuga de datos; las métricas reportadas son una estimación honesta del desempeño en pacientes nuevos.
3. **Sensibilidad como prioridad clínica:** en tamizaje, un falso negativo (neumonía no detectada) es más costoso que un falso positivo. Los pesos de clase y el umbral ajustado en validación equilibran sensibilidad y especificidad.
4. **Explicabilidad:** los mapas CAM permiten verificar que el modelo atiende a los campos pulmonares y detectar posibles atajos.
5. **Limitaciones:** población pediátrica de un único centro hospitalario, proyección anteroposterior y etiquetas sin confirmación de laboratorio. El modelo es una herramienta de apoyo académico y **no** sustituye el criterio radiológico; su uso clínico exigiría validación externa.

## Referencias (APA 7)
- Kermany, D. S., Goldbaum, M., Cai, W., Valentim, C. C. S., Liang, H., Baxter, S. L., ... Zhang, K. (2018). Identifying medical diagnoses and treatable diseases by image-based deep learning. *Cell, 172*(5), 1122–1131. https://doi.org/10.1016/j.cell.2018.02.010
- Mooney, P. (2018). *Chest X-Ray Images (Pneumonia)* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. En *Proceedings of the IEEE International Conference on Computer Vision* (pp. 618–626).
- Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking model scaling for convolutional neural networks. En *Proceedings of the 36th International Conference on Machine Learning* (pp. 6105–6114). PMLR.
- Zhou, B., Khosla, A., Lapedriza, A., Oliva, A., & Torralba, A. (2016). Learning deep features for discriminative localization. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition* (pp. 2921–2929).